In [1]:
import json
import requests

MUSE_BASE_URL = "https://www.themuse.com/api/public/jobs"

# Single test call — no auth needed
response = requests.get(MUSE_BASE_URL, params={
    "page": 1,
    "descending": "true",
})
response.raise_for_status()

data = response.json()

print(f"Total results available: {data.get('total')}")
print(f"Results this page:       {len(data.get('results', []))}")
print(f"\nTop-level keys: {list(data.keys())}")
print(f"\n--- First result raw ---")
print(json.dumps(data['results'][0], indent=2))

Total results available: 499177
Results this page:       20

Top-level keys: ['page', 'page_count', 'items_per_page', 'took', 'timed_out', 'total', 'results', 'aggregations']

--- First result raw ---
{
  "contents": "<p>Responsibilities<br><br>1. Monitor remotely, critical infrastructure equipment which supports IT systems and components in the electrical distribution through an integrated platform, followed by relevant incident and problem management<br><br>2. Responsible for answering incoming telephone calls, alerts, and web-based incidents and prioritising customer support needs<br><br>3. Manage and maintain critical facility Monitoring Systems (monitoring and management of air-conditioning, UPS, generators, power usage, running health and safety and housekeeping checks)<br><br>4. Awareness and alignment to internal OLAs, external SLAs and personal KPIs<br><br>5. Understanding of Energy Efficiency concepts and identifying opportunities to optimise energy consumption patterns.<br><

In [2]:
# Test category filtering — these are the documented category values
categories_to_test = [
    "Data Science",
    "Software Engineer", 
    "Data and Analytics",
]

for cat in categories_to_test:
    r = requests.get(MUSE_BASE_URL, params={
        "category": cat,
        "page": 1,
    })
    data = r.json()
    count = data.get("total", 0)
    print(f"category='{cat}' → {count} total results")
    if data.get("results"):
        print(f"  Sample: {data['results'][0].get('name', '')} @ "
              f"{data['results'][0].get('company', {}).get('name', '')}")
    print()

category='Data Science' → 2 total results

category='Software Engineer' → 13 total results

category='Data and Analytics' → 19276 total results
  Sample: Director, Transformation FP&A @ Disney



In [6]:
from openai import OpenAI
from langsmith.wrappers import wrap_openai
from dotenv import load_dotenv
import os

load_dotenv()
client = wrap_openai(OpenAI(api_key=os.getenv("OPENAI_API_KEY")))

Cell 1: Calling muse api and stripping html

In [7]:
from bs4 import BeautifulSoup
from datetime import datetime

MUSE_BASE_URL = "https://www.themuse.com/api/public/jobs"

MUSE_LEVEL_MAP = {
    "Internship":   ["Internship"],
    "Entry-level":  ["Entry Level", "Internship"],
    "Mid-level":    ["Mid Level"],
    "Senior-level": ["Senior Level", "Management", "Director"],
    "Director":     ["Director", "VP", "Executive"],
}

MUSE_CATEGORY_MAP = {
    "data engineer":     "Data and Analytics",
    "ml engineer":       "Data Science",
    "ai engineer":       "Data Science",
    "data scientist":    "Data Science",
    "data analyst":      "Data and Analytics",
    "software engineer": "Software Engineering",
    "backend engineer":  "Software Engineering",
    "frontend engineer": "Software Engineering",
}

def _infer_muse_category(job_titles: str) -> str:
    titles_lower = job_titles.lower()
    for keyword, category in MUSE_CATEGORY_MAP.items():
        if keyword in titles_lower:
            return category
    return "Data Science"

def _strip_html(html: str) -> str:
    """Strip HTML tags from Muse contents field, return clean text."""
    if not html:
        return ""
    return BeautifulSoup(html, "html.parser").get_text(separator=" ", strip=True)


def search_jobs_muse(
    job_titles: str,
    location: str,
    experience_level: str,
    results_per_page: int = 10,
) -> list[dict]:
    category = _infer_muse_category(job_titles)
    levels   = MUSE_LEVEL_MAP.get(experience_level, [])

    # Build params — requests serializes list values as repeated keys
    # which is what the Muse API expects for multiple levels
    params = [
        ("category",   category),
        ("location",   location),
        ("page",       1),
        ("descending", "true"),
    ]
    for level in levels:
        params.append(("level", level))

    response = requests.get(MUSE_BASE_URL, params=params)
    response.raise_for_status()

    raw_results = response.json().get("results", [])[:results_per_page]

    jobs = []
    for job in raw_results:
        # Skip if no URL — nothing we can do with it
        url = job.get("refs", {}).get("landing_page", "")
        if not url:
            continue

        # Skip if no description content
        raw_contents = job.get("contents", "")
        description  = _strip_html(raw_contents)
        if not description:
            continue

        locations    = job.get("locations", [])
        location_str = locations[0].get("name", "") if locations else ""

        levels_raw   = job.get("levels", [])
        level_str    = levels_raw[0].get("name", "") if levels_raw else ""

        jobs.append({
            "title":       job.get("name", ""),
            "company":     job.get("company", {}).get("name", ""),
            "location":    location_str,
            "level":       level_str,
            "salary_min":  None,
            "salary_max":  None,
            "description": description,
            "url":         url,
            "created":     job.get("publication_date", datetime.utcnow().isoformat()),
        })

    return jobs

Cell 2: 

In [8]:
def muse_intent_llm(
    experience_level: str,
    job_titles: str,
    state: str,
    notes: str,
    results_per_page: int = 10,
) -> list[dict]:
    """
    Takes the same 4 search inputs as intent_llm().
    Uses a lightweight LLM call to extract the best Muse search parameters,
    then calls search_jobs_muse() and returns normalized job dicts.
    """

    # Tell the LLM what categories and locations Muse actually supports
    # so it picks valid values rather than hallucinating
    prompt = f"""
You are a search parameter extractor. Given a user's job search inputs, extract the optimal parameters to search The Muse job API.

The Muse API supports these categories (pick exactly one, the most relevant):
- "Data and Analytics"
- "Data Science"  
- "Software Engineering"
- "Engineering"
- "Product"

The Muse API location field accepts city or region strings like:
"New York City, New York", "San Francisco, California", "Remote", "Chicago, Illinois"
Convert the user's state input into a clean city + state string using the most relevant major city for that state.

User inputs:
Experience Level: {experience_level}
Job Titles: {job_titles}
State: {state}
Notes: {notes if notes.strip() else "None provided"}

Return ONLY valid JSON:
{{
    "category": "the single best matching Muse category",
    "location": "City, State format",
    "reasoning": "one sentence on why these parameters fit the user's search"
}}
"""

    for attempt in range(2):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                max_tokens=200,
                response_format={"type": "json_object"},
            )
            params = json.loads(response.choices[0].message.content)

            print(f"[muse_intent_llm] category='{params.get('category')}' "
                  f"location='{params.get('location')}'")
            print(f"[muse_intent_llm] reasoning: {params.get('reasoning')}")

            jobs = search_jobs_muse(
                job_titles=job_titles,
                location=params.get("location", state),
                experience_level=experience_level,
                results_per_page=results_per_page,
            )

            return jobs

        except Exception as e:
            if attempt == 1:
                print(f"[muse_intent_llm] Failed after retry: {e}")
                return []

Cell 3: End to end test

In [9]:
muse_jobs = muse_intent_llm(
    experience_level="Entry-level",
    job_titles="Data Engineer, ML Engineer",
    state="New York",
    notes="prefer AI-first companies, interested in LLM tooling",
    results_per_page=10,
)

print(f"\nJobs returned: {len(muse_jobs)}\n")
for i, job in enumerate(muse_jobs, 1):
    print(f"{i}. {job['title']} @ {job['company']}")
    print(f"   Level:       {job['level']}")
    print(f"   Location:    {job['location']}")
    print(f"   URL:         {job['url']}")
    print(f"   Description: {len(job['description'])} chars")
    print(f"   Preview:     {job['description'][:300]!r}")
    print()

[muse_intent_llm] category='Data Science' location='New York City, New York'
[muse_intent_llm] reasoning: Data Engineer and ML Engineer roles are closely aligned with the Data Science category, which also encompasses interests in AI and LLM tooling.

Jobs returned: 4

1. Treasury Intern @ Equinix, Inc
   Level:       Internship
   Location:    Flexible / Remote
   URL:         https://www.themuse.com/jobs/equinixinc/treasury-intern
   Description: 3810 chars
   Preview:     "Who are we? Equinix is the world's digital infrastructure company®, shortening the path to connectivity to enable the innovations that enrich our work, life and planet. A place where bold ideas are welcomed, human connection is valued, and everyone has the opportunity to shape their future. A career"

2. AI Applied Scientist - PhD Intern, Foundational IQ @ Zillow
   Level:       Internship
   Location:    Flexible / Remote
   URL:         https://www.themuse.com/jobs/zillow/ai-applied-scientist-phd-intern-foundatio

C:\Users\19083\AppData\Local\Temp\ipykernel_57144\706475496.py:92: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created":     job.get("publication_date", datetime.utcnow().isoformat()),


**Full Intent_llm with adzuna and the muse**

In [13]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
from langsmith.wrappers import wrap_openai

load_dotenv()
client = wrap_openai(OpenAI(api_key=os.getenv("OPENAI_API_KEY")))

# ── Constants ─────────────────────────────────────────────────────────────────

INTENT_SYSTEM_PROMPT = """
You are a job search query planner. Your job is to take structured search inputs from a user and produce an optimal search plan across two platforms: Adzuna (a job board API) and The Muse (a job API).

HARD CONSTRAINTS — never violate these:
- adzuna_queries: maximum 3 items, minimum 1
- Every Adzuna query must include location — never omit it
- Adzuna keywords should be concise (3-6 words), location should be the city or state
- Adzuna keyword strings must NEVER include the experience level — do NOT write "entry level data engineer", just write "data engineer". Experience level is passed separately to the API.
- For muse_params: pick exactly one category from this list:
    "Data and Analytics", "Data Science", "Software Engineering", "Engineering", "Product"
- For muse_params: convert the user's state into a clean "City, State" string using the most relevant major city for that state
- Queries must be specific and targeted, never generic
- If multiple job titles are provided, spread them across Adzuna queries rather than cramming all titles into one query

Return ONLY valid JSON with exactly these keys:
{
    "adzuna_queries": [{"keywords": "...", "location": "..."}],
    "muse_params": {
        "category": "...",
        "location": "City, State"
    },
    "search_reasoning": "one sentence explaining the overall query strategy"
}
""".strip()

# ── Intent LLM ────────────────────────────────────────────────────────────────

def intent_llm(
    experience_level: str,
    job_titles: str,
    state: str,
    notes: str,
) -> dict:
    user_message = f"""
Experience Level: {experience_level}
Job Titles: {job_titles}
State: {state}
Notes: {notes if notes.strip() else "None provided"}
""".strip()

    for attempt in range(2):
        try:
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": INTENT_SYSTEM_PROMPT},
                    {"role": "user",   "content": user_message},
                ],
                max_tokens=500,
                response_format={"type": "json_object"},
            )
            result = json.loads(response.choices[0].message.content)

            assert "adzuna_queries"   in result
            assert "muse_params"      in result
            assert "search_reasoning" in result
            assert "category" in result["muse_params"]
            assert "location" in result["muse_params"]

            # Hard cap — nothing else
            result["adzuna_queries"] = result["adzuna_queries"][:3]

            return result

        except (json.JSONDecodeError, AssertionError, Exception) as e:
            if attempt == 1:
                print(f"[intent_llm] Failed after retry: {e}")
                return {}

# ── Test ──────────────────────────────────────────────────────────────────────

plan = intent_llm(
    experience_level="Entry-level",
    job_titles="Data Engineer, ML Engineer",
    state="New York",
    notes="prefer AI-first companies, interested in LLM tooling",
)

print(json.dumps(plan, indent=2))

print("\n--- Adzuna queries ---")
for i, q in enumerate(plan.get("adzuna_queries", []), 1):
    print(f"  Query {i}: keywords='{q['keywords']}' | location='{q['location']}'")

print("\n--- Muse params ---")
mp = plan.get("muse_params", {})
print(f"  category: {mp.get('category')}")
print(f"  location: {mp.get('location')}")

print(f"\n--- Reasoning ---")
print(f"  {plan.get('search_reasoning')}")

{
  "adzuna_queries": [
    {
      "keywords": "data engineer",
      "location": "New York, NY"
    },
    {
      "keywords": "ML engineer",
      "location": "New York, NY"
    }
  ],
  "muse_params": {
    "category": "Data Science",
    "location": "New York, NY"
  },
  "search_reasoning": "The strategy is to target both specified job titles in Adzuna while also focusing on Data Science roles on The Muse to find opportunities at AI-first companies."
}

--- Adzuna queries ---
  Query 1: keywords='data engineer' | location='New York, NY'
  Query 2: keywords='ML engineer' | location='New York, NY'

--- Muse params ---
  category: Data Science
  location: New York, NY

--- Reasoning ---
  The strategy is to target both specified job titles in Adzuna while also focusing on Data Science roles on The Muse to find opportunities at AI-first companies.
